In [ ]:
import pandas as pd

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


analytics_mongo = client_db.get_database('analytics_db')

db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')

wave=app_mongo.get_database('app_db')



In [ ]:
vehicle_number_list=list(pd.read_sql("select vehicle_number from vehicle_details ",engine)['vehicle_number'])

In [ ]:
len(vehicle_number_list)

In [ ]:
vahan_data=pd.DataFrame()
for i in range(0,350000,10000):
    vahan_data_reuse=pd.DataFrame(list(db.vahaan.find({'registration_no':{'$in':vehicle_number_list[i:i+10000]}},{'maker_model':1,'registration_no':1})))  
    vahan_data=pd.concat([vahan_data,vahan_data_reuse])
    print(i)


In [ ]:
make_model=vahan_data

In [ ]:
make_model.registration_no.nunique()

In [ ]:
make_model['type']=make_model['maker_model'].apply(lambda x: str(type(x)))
make_model=make_model[make_model['type']!="<class 'datetime.datetime'>"]
del make_model['type']

make_model['contains_upto']=make_model['maker_model'].str.findall('upto').apply(len)
make_model=make_model[make_model['contains_upto']==0]
del make_model['contains_upto']

make_model['make']=make_model['maker_model'].apply(lambda x : x.split('/')[0])


make_model['contains_numbers']=make_model['make'].str.findall(r'\d\d').apply(len)
make_model=make_model[make_model['contains_numbers']==0]
del make_model['contains_numbers']

In [ ]:
make_model

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [ ]:
def OEM_categorisation(x):
    name_list=[ 'ASHOK LEYLAND',
        'TATA',
 'VE COMMERCIAL',
 'MAHINDRA',
 'EICHER',
 'DAIMLER',
 'SML ISUZU',
 'VECTRA',
 'Baldev',
 'FORCE',
 'PIAGGIO',
 'MAN TRUCKS',
 'AMW',
 'GENERAL']
    for i in name_list:
        if fuzz.partial_ratio(x,i)>70:
            #print('compared ' + i + ' with '+ x +' and SELECTED')
            return i 
        else:
            None
            #print('compared ' + i + ' with '+ x +' and rejected')
    return x
                
    

In [ ]:
make_model=make_model[~make_model['make'].isin([' ','  ','   ','    ','     '])]

In [ ]:
make_model['sanitized_make']=make_model.make.apply(OEM_categorisation)

In [ ]:
def split_data(x):
    try:
        a=x.split('/')[1:]
        b=''
        for i in a:
            b=b+i
        return b
    except:
        a=None
        return a

In [ ]:
make_model['model']=make_model['maker_model'].apply(split_data)


In [ ]:
make_model=make_model[make_model['model'].notnull()]

In [ ]:
grid=pd.read_csv("Grid - Sheet1.csv")

In [ ]:
total=''
for i in grid[grid['type'].isin(['4 digs'])]['correction'].unique():
    a=i[0:2]+'\D'+i[2:4]
    total=total+"|"+a
    total=total+"|"+i


In [ ]:
for i in grid[grid['type'].isin(['3 digs','3 dig','5 dig'])]['correction'].unique():
    total=total+"|"+i


In [ ]:
for i in grid[grid['type'].isin(['text'])]['correction'].unique():
    total=total+"|"+"(?i)"+i


In [ ]:
exclusive_models=['torro25','traco35','truxo25','truxo31','torro.25','traco.35','truxo.25','truxo.31','Blazo37','Blazo25','Blazo40','Blazo49','Blazo.37','Blazo.25','Blazo.40','Blazo.49','load.king','loadking','pickup','pick.up']    



In [ ]:
for i in exclusive_models:
    total=total+"|"+"(?i)"+i

    

In [ ]:
make_model['model_regex']=make_model['model'].str.findall('r'+str(total[1:]))


In [ ]:
make_model['model_regex']=make_model['model_regex'].apply(lambda x: x if len(x)>0 else None)

In [ ]:
import re
def model_create(model):
    if model!=None:
        final=''
        for i in model: 
            i = re.sub('\.', '', i)
            i= re.sub('-', '', i)
            i= re.sub(' ', '', i)
            i= re.sub(':', '', i)
            i= re.sub('\*', '', i)
            i=i.upper()
            if i not in final.split(' '):
                final=final+" "+i
        alphabatecally=''
        for i in reversed(sorted(final.split(' '))):
            alphabatecally=alphabatecally+" "+i
            
        return alphabatecally[1:]
    else:
        return None

        
    

In [ ]:
make_model['model_type']=make_model['model_regex'].apply(model_create)

In [ ]:
make_model.model_type.isna().value_counts()


In [ ]:
make_model['make_model_sanitized']=make_model['sanitized_make']+' '+make_model['model_type']

In [ ]:
make_model=make_model.drop_duplicates('registration_no')

In [ ]:
make_model.model_type.isna().value_counts()

In [ ]:
for i in range (0,310000,10000):
    make_model[i:i+5000].set_index(['registration_no']).to_sql("vehicle_make_model",con=engine,if_exists='append',chunksize=5000,method='multi')  
    print(i)

In [ ]:
pd.read_sql("select *  from vehicle_make_model limit 100",engine)

In [ ]:
make_model[i:i+5000]

In [ ]:
i